In [2]:
##### SIF #####
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

IN_DIR  = Path(r"C:\Drought\Regridding and data clipping\SIF")
IN_FILE = IN_DIR / "GOSIF_SIF_India_0.05deg_2001-2023.nc"
OUT_FILE = IN_DIR / "SIF_Z_monthly_India_0p05deg_2001present.nc"

BASE_START = "2001-01"
BASE_END   = "2020-12"
CHUNKS = {"time": 36, "lat": 300, "lon": 300}

def to_month_end(da):
    t = pd.to_datetime(da["time"].values)
    mend = pd.PeriodIndex(t, freq="M").to_timestamp("M")
    da2 = da.assign_coords(time=mend)
    _, idx = np.unique(da2["time"].values, return_index=True)
    return da2.isel(time=np.sort(idx)).sortby("time")

def pick_var(ds, prefs):
    for p in prefs:
        if p in ds:
            return ds[p]
    if len(ds.data_vars) == 1:
        return ds[list(ds.data_vars)[0]]
    raise ValueError(f"Could not find any of {prefs} in dataset; available={list(ds.data_vars)}")

def monthwise_stats(da, base_start, base_end):
    base = da.sel(time=slice(base_start, base_end))
    mus, sds = [], []
    for m in range(1, 13):
        bm = base.where(base["time.month"] == m, drop=True)
        mus.append(bm.mean("time", skipna=True))
        sds.append(bm.std("time", ddof=1, skipna=True))
    mu = xr.concat(mus, dim="month").assign_coords(month=np.arange(1,13))
    sd = xr.concat(sds, dim="month").assign_coords(month=np.arange(1,13))
    return mu, sd

def apply_month_index(da, aux_by_month):
    idx = xr.DataArray(da["time.month"].values, coords={"time": da["time"]}, dims=("time",))
    return aux_by_month.sel(month=idx)

print("Loading SIF …")
ds = xr.open_dataset(IN_FILE)
sif = pick_var(ds, ["SIF", "sif", "GOSIF"]).rename("SIF")
sif = to_month_end(sif).chunk(CHUNKS)

print("Baseline stats …")
mu_m, sd_m = monthwise_stats(sif, BASE_START, BASE_END)
mu_t = apply_month_index(sif, mu_m)
sd_t = apply_month_index(sif, sd_m)

print("Anomaly & Z …")
SIF_anom = (sif - mu_t).rename("SIF_anom")
SIF_z = ((sif - mu_t) / sd_t).clip(-3, 3).rename("SIF_z").assign_attrs(units="1")

print(f"Writing {OUT_FILE} …")
ds_out = xr.Dataset(
    data_vars=dict(
        SIF=sif,
        SIF_anom=SIF_anom,
        SIF_z=SIF_z,
    ),
    attrs=dict(
        title="Monthly SIF anomalies and Z-scores (0.05°, India)",
        summary="Calendar-month standardized anomalies vs 2001–2020.",
        conventions="CF-1.8",
        crs="EPSG:4326",
        spatial_resolution="0.05 degree",
        temporal_aggregation="monthly",
        history="Generated by make_sif_z.py",
        baseline=f"{BASE_START}..{BASE_END}",
    ),
)
comp = dict(zlib=True, complevel=4)
encoding = {v: comp for v in ds_out.data_vars}
encoding.update({"lat": {"dtype": "float32"}, "lon": {"dtype": "float32"}})
ds_out.to_netcdf(OUT_FILE, encoding=encoding)
print("Done.")


Loading SIF …


C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


Baseline stats …
Anomaly & Z …
Writing C:\Drought\Regridding and data clipping\SIF\SIF_Z_monthly_India_0p05deg_2001present.nc …


C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\dask\array\numpy_compat.py:57: RuntimeWarning: invalid value encountered in divide
  x = np.divide(x1, x2, out)
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\dask\core.py:127: RuntimeWarning: divide by zero encountered in divide
  return func(*(_execute_task(a, cache) for a in args))
C:\Users\Vikas.Patel\AppData\Local\anaconda3\Lib\site-packages\dask\core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


Done.
